手写LORA


In [77]:
import transformers
import peft
import datasets
import tqdm
import einops
import torch
from torch import nn
import torch.functional as F
import copy

In [78]:
class MyLora(nn.Module):
    def __init__(self,base_layer: nn.Linear,alpha:int = 1, lora_rank:int=8, dropout_p:float=0.0, test_mode: bool = False):
        
        super(MyLora,self).__init__()
        self.base_layer = copy.deepcopy(base_layer)
        self.lora_rank = lora_rank
        self.alpha = alpha
        self.test_mode = test_mode
        self.A = nn.Parameter(torch.empty((lora_rank, base_layer.in_features),dtype=base_layer.weight.dtype))
        self.B = nn.Parameter(torch.empty((base_layer.out_features, lora_rank),dtype=base_layer.weight.dtype))
        self.dropout = nn.Dropout(dropout_p)
        # 初始化 lora 矩阵
        nn.init.normal_(self.A, mean=0.0, std=0.02)
        if test_mode:
            nn.init.normal_(self.B, mean=0.0, std=0.02)
        else:
            nn.init.zeros_(self.B)

        # 冻结原来的层的参数
        for param in self.base_layer.parameters():
            param.requires_grad = False

    def forward(self, x):
        scaling = float(self.alpha) / float(self.r)
        h = F.Linear(self.dropout(x),self.A)
        output = F.Linear(h,self.B)
        return self.base_layer(x) + output * scaling

替换

In [79]:
def replace_lora(model, lora_rank=8, alpha=1, dropout_p=0.1, test_mode=False, 
    embed_requires_grad: bool = False,      # embedding 层是否训练
    norm_requires_grad: bool = False,       # norm 层是否训练
    head_requires_grad: bool = False,       # lm_head 层是否训练（Causal LM才有）
    ):
    for name, module in model.named_children():
        if any(s in name for s in ['embed', 'norm', 'lm_head']):
            requires_grad = embed_requires_grad if 'embed' in name \
                            else norm_requires_grad if 'norm' in name \
                            else head_requires_grad
            for param in module.parameters():
                param.requires_grad = requires_grad
        elif isinstance(module, nn.Linear) and module.weight.requires_grad:
            new_module = MyLora(module, lora_rank=lora_rank, alpha=alpha, dropout_p=dropout_p, test_mode=test_mode)
            setattr(model, name, new_module)
        else:
            replace_lora(module, lora_rank=lora_rank, alpha=alpha, dropout_p=dropout_p, test_mode=test_mode,embed_requires_grad=embed_requires_grad,
                norm_requires_grad=norm_requires_grad, head_requires_grad=head_requires_grad)

In [80]:
def print_trainable_parameters(model:nn.Module):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    percent = trainable_params / total_params * 100 if total_params > 0 else 0
    print(f"Total parameters: {total_params}, Trainable parameters: {trainable_params}, "
          f"Trainable percentage: {percent:.2f}%")

验证一下

In [81]:
from transformers import AutoConfig

config = AutoConfig.for_model('llama')
config.hidden_size = 24
config.intermediate_size = config.hidden_size * 4
config.num_attention_heads = 4
config.num_hidden_layers = 4
config.num_key_value_heads = 2
config.vocab_size = 128

In [82]:
from transformers import AutoModel, AutoModelForCausalLM

raw_model = AutoModel.from_config(config)  # 没带因果头
# raw_model = AutoModelForCausalLM.from_config(config)  # 带了因果头
print(raw_model)

"""
LlamaModel(
  (embed_tokens): Embedding(128, 24)
  (layers): ModuleList(
    (0-3): 4 x LlamaDecoderLayer(
      (self_attn): LlamaSdpaAttention(
        (q_proj): Linear(in_features=24, out_features=24, bias=False)
        (k_proj): Linear(in_features=24, out_features=12, bias=False)
        (v_proj): Linear(in_features=24, out_features=12, bias=False)
        (o_proj): Linear(in_features=24, out_features=24, bias=False)
        (rotary_emb): LlamaRotaryEmbedding()
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=24, out_features=96, bias=False)
        (up_proj): Linear(in_features=24, out_features=96, bias=False)
        (down_proj): Linear(in_features=96, out_features=24, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm()
      (post_attention_layernorm): LlamaRMSNorm()
    )
  )
  (norm): LlamaRMSNorm()
)
"""

LlamaModel(
  (embed_tokens): Embedding(128, 24)
  (layers): ModuleList(
    (0-3): 4 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=24, out_features=512, bias=False)
        (k_proj): Linear(in_features=24, out_features=256, bias=False)
        (v_proj): Linear(in_features=24, out_features=256, bias=False)
        (o_proj): Linear(in_features=512, out_features=24, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=24, out_features=96, bias=False)
        (up_proj): Linear(in_features=24, out_features=96, bias=False)
        (down_proj): Linear(in_features=96, out_features=24, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm((24,), eps=1e-06)
      (post_attention_layernorm): LlamaRMSNorm((24,), eps=1e-06)
    )
  )
  (norm): LlamaRMSNorm((24,), eps=1e-06)
  (rotary_emb): LlamaRotaryEmbedding()
)


'\nLlamaModel(\n  (embed_tokens): Embedding(128, 24)\n  (layers): ModuleList(\n    (0-3): 4 x LlamaDecoderLayer(\n      (self_attn): LlamaSdpaAttention(\n        (q_proj): Linear(in_features=24, out_features=24, bias=False)\n        (k_proj): Linear(in_features=24, out_features=12, bias=False)\n        (v_proj): Linear(in_features=24, out_features=12, bias=False)\n        (o_proj): Linear(in_features=24, out_features=24, bias=False)\n        (rotary_emb): LlamaRotaryEmbedding()\n      )\n      (mlp): LlamaMLP(\n        (gate_proj): Linear(in_features=24, out_features=96, bias=False)\n        (up_proj): Linear(in_features=24, out_features=96, bias=False)\n        (down_proj): Linear(in_features=96, out_features=24, bias=False)\n        (act_fn): SiLU()\n      )\n      (input_layernorm): LlamaRMSNorm()\n      (post_attention_layernorm): LlamaRMSNorm()\n    )\n  )\n  (norm): LlamaRMSNorm()\n)\n'

In [83]:
print_trainable_parameters(raw_model)

Total parameters: 178392, Trainable parameters: 178392, Trainable percentage: 100.00%


In [ ]:
lora_model =  copy.deepcopy(raw_model)  # 深克隆，独立一个新模型
replace_lora(lora_model, lora_rank=8, alpha=16)  # 替换
print_trainable_parameters(lora_model) # 打印参数情况
print(lora_model)

"""
trainable params: 16,896 || all params: 54,744 || trainable%: 30.8637

LlamaModel(
  (embed_tokens): Embedding(128, 24)
  (layers): ModuleList(
    (0-3): 4 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): LoraLinear(
          (base_layer): Linear(in_features=24, out_features=24, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (k_proj): LoraLinear(
          (base_layer): Linear(in_features=24, out_features=12, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (v_proj): LoraLinear(
          (base_layer): Linear(in_features=24, out_features=12, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (o_proj): LoraLinear(
          (base_layer): Linear(in_features=24, out_features=24, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (rotary_emb): LlamaRotaryEmbedding()
      )
      (mlp): LlamaMLP(
        (gate_proj): LoraLinear(
          (base_layer): Linear(in_features=24, out_features=96, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (up_proj): LoraLinear(
          (base_layer): Linear(in_features=24, out_features=96, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (down_proj): LoraLinear(
          (base_layer): Linear(in_features=96, out_features=24, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm()
      (post_attention_layernorm): LlamaRMSNorm()
    )
  )
  (norm): LlamaRMSNorm()
)
"""

Total parameters: 242136, Trainable parameters: 63744, Trainable percentage: 26.33%
LlamaModel(
  (embed_tokens): Embedding(128, 24)
  (layers): ModuleList(
    (0-3): 4 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): MyLora(
          (base_layer): Linear(in_features=24, out_features=512, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (k_proj): MyLora(
          (base_layer): Linear(in_features=24, out_features=256, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (v_proj): MyLora(
          (base_layer): Linear(in_features=24, out_features=256, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (o_proj): MyLora(
          (base_layer): Linear(in_features=512, out_features=24, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (mlp): LlamaMLP(
        (gate_proj): MyLora(
          (base_layer): Linear(in_features=24, out_feature

'\ntrainable params: 16,896 || all params: 54,744 || trainable%: 30.8637\n\nLlamaModel(\n  (embed_tokens): Embedding(128, 24)\n  (layers): ModuleList(\n    (0-3): 4 x LlamaDecoderLayer(\n      (self_attn): LlamaAttention(\n        (q_proj): LoraLinear(\n          (base_layer): Linear(in_features=24, out_features=24, bias=False)\n          (dropout): Dropout(p=0.0, inplace=False)\n        )\n        (k_proj): LoraLinear(\n          (base_layer): Linear(in_features=24, out_features=12, bias=False)\n          (dropout): Dropout(p=0.0, inplace=False)\n        )\n        (v_proj): LoraLinear(\n          (base_layer): Linear(in_features=24, out_features=12, bias=False)\n          (dropout): Dropout(p=0.0, inplace=False)\n        )\n        (o_proj): LoraLinear(\n          (base_layer): Linear(in_features=24, out_features=24, bias=False)\n          (dropout): Dropout(p=0.0, inplace=False)\n        )\n        (rotary_emb): LlamaRotaryEmbedding()\n      )\n      (mlp): LlamaMLP(\n        (gate_

查看是不是只有Lora层是可训练

In [85]:
def print_model_parameters(model):
    """
    查看模型参数的 requires_grad 情况
    """
    print("Layer Name & Parameters")
    print("----------------------------")
    for name, parameter in model.named_parameters():
        print(f"{name:50} | Requires_grad: {parameter.requires_grad}")

In [86]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules='all-linear', # 太低版本的 peft 不支持这种做法
)
peft_lora_model = copy.deepcopy(raw_model)
peft_lora_model = get_peft_model(peft_lora_model, lora_config)
peft_lora_model.print_trainable_parameters()

"""
trainable params: 16,896 || all params: 54,744 || trainable%: 30.8637
"""

trainable params: 63,744 || all params: 242,136 || trainable%: 26.3257


'\ntrainable params: 16,896 || all params: 54,744 || trainable%: 30.8637\n'

卸载和重载